# 00 · Yambda-50M の取得と検証

Notion本文を正とし、50Mのflat再生・いいねのみ取得します。公式リビジョンとSHA256を固定します。全件をpandasには読みません。

**時刻の注意**：公式コードはtimestampを秒で扱っています。値は5秒刻みですが、5倍はしません。本文の約301日と一致する `timestamp // 86400` を使います。詳細は `DECISIONS.md`。

In [2]:
import sys
import polars as pl

print("Python:", sys.executable)
print("Polars:", pl.__version__)

Python: /Users/macuser/dev/paper1/.venv/bin/python
Polars: 1.44.2


In [3]:
from pathlib import Path
import numpy as np
import polars as pl
from IPython.display import display, Image
from relisten import config
np.random.seed(config.SEED)
print("作業フォルダ:", config.ROOT)

作業フォルダ: /Users/macuser/dev/paper1


In [4]:
from relisten.data import prepare_data, embedding_coverage
summary, manifest = prepare_data()
display(summary)
display(manifest)

listens,users,items,min_timestamp,max_timestamp,organic_fraction,valid_routes,has_nulls
u32,u32,u32,u32,u32,f64,bool,bool
46467212,9238,877168,0,26000000,0.516572,true,false


{'repo': 'yandex/yambda',
 'revision': 'dd6f3a19eef5866e346c3270e098baa641a44948',
 'files': {'flat/50m/listens.parquet': {'size': 369110332,
   'sha256': 'eed9cbd094af1e189507d2f8132a0dc9653b90e65480125c7cdccd601e0592d1'},
  'flat/50m/likes.parquet': {'size': 7180817,
   'sha256': '694087077dbacfcc1d22a5ca85cc6bd8ab182361933406e60500624c6a422bc4'}}}

## 次のnotebookに渡すファイル

`items_50m.parquet`、`user_days.parquet`、`representation_train.parquet` をdata/processedに保存します。最後のファイルは149日目までに物理的に限定した表現学習専用の再生ログです。

In [5]:
display(pl.scan_parquet(config.PROCESSED / "representation_train.parquet").select(pl.len(), pl.col("timestamp").max().alias("max_timestamp")).collect())

len,max_timestamp
u32,u32
19537609,12959995


## 音響埋め込みの収録率

全曲分の約13.8GBは取得せず、HTTP rangeでitem_id列だけを読みます。50Mの曲と重なるIDだけを保存します。ベクトル自体は今回のEDAには不要です。

In [6]:
display(embedding_coverage(manifest["revision"]))

items,embedded_items,fraction,revision
i64,i64,f64,str
877168,819432,0.934179,"""dd6f3a19eef5866e346c3270e098ba…"


In [7]:
from relisten import config

print(config.ROOT)
print(list((config.ROOT / "data").rglob("*.parquet"))[:20])

/Users/macuser/dev/paper1
[PosixPath('/Users/macuser/dev/paper1/data/processed/items_50m.parquet'), PosixPath('/Users/macuser/dev/paper1/data/processed/embedding_item_ids_50m.parquet'), PosixPath('/Users/macuser/dev/paper1/data/processed/recent7d_features.parquet'), PosixPath('/Users/macuser/dev/paper1/data/processed/SASRec_e2e_test_l7.parquet'), PosixPath('/Users/macuser/dev/paper1/data/processed/SASRec_e2e_validation_l7.parquet'), PosixPath('/Users/macuser/dev/paper1/data/processed/actr_base_variant_features.parquet'), PosixPath('/Users/macuser/dev/paper1/data/processed/BPR_features.parquet'), PosixPath('/Users/macuser/dev/paper1/data/processed/main_labels.parquet'), PosixPath('/Users/macuser/dev/paper1/data/processed/pairs.parquet'), PosixPath('/Users/macuser/dev/paper1/data/processed/SASRec_e2e_validation_l14.parquet'), PosixPath('/Users/macuser/dev/paper1/data/processed/SASRec_features.parquet'), PosixPath('/Users/macuser/dev/paper1/data/processed/SASRec_e2e_validation_l3.parquet'

In [8]:
import polars as pl

path = list((config.ROOT / "data").rglob("*.parquet"))[0]

df = pl.scan_parquet(path)

display(df.head(10).collect())

item_id
u32
8958234
558384
8925286
3542184
4414982
3491662
7732362
7582253
5009703


In [9]:
# 公式配布ファイル（Yambda-50M flat）の先頭行
for name in ["listens", "likes"]:
    display(pl.scan_parquet(config.RAW / f"flat/50m/{name}.parquet").head(10).collect())


uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds
u32,u32,u32,u8,u16,u32
100,39420,8326270,0,100,170
100,39420,1441281,0,100,105
100,39625,286361,0,100,185
100,40110,732449,0,100,240
100,40360,3397170,0,46,130
100,40380,7849270,0,100,205
100,40640,1449307,0,100,205
100,40785,6887232,0,100,145
100,40885,2996197,0,100,95


uid,timestamp,item_id,is_organic
u32,u32,u32,u8
100,44755,732449,1
100,1155860,6568592,0
100,1259125,5411243,1
100,1260005,7371186,0
100,1263935,4943655,0
100,2055015,410868,1
100,2494620,1730273,1
100,3939350,2621343,0
100,3940755,6399268,0
